In [11]:

import os
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict

from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.messages import SystemMessage
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain.messages import AIMessage,HumanMessage,SystemMessage

In [6]:
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b")

In [7]:
class State(TypedDict):
    messages:Annotated[list,add_messages]

In [ ]:
def chatnode(state:State):
    # interupt will pause the graphs execution
    decisions=interrupt({
        "type":"approval", # identifies tht this is an approval request
        "reason":"model is about to answer users question", # why approval is needed
        "question":state['messages'][-1].content, # to get the question
        "instructions":"approve this question (y/n)?" #tells the human what decision is expected.
    })

    if decisions['approved']=='n':
        return {
            "messages":[AIMessage(content="Not Approved.")]
        }
    else:
        return {
            "messages":[llm.invoke(state['messages'])]
        }

In [12]:
builder=StateGraph(State)

builder.add_node('chatnode',chatnode)
builder.add_edge(START,'chatnode')
builder.add_edge('chatnode',END)

memory=InMemorySaver()

graph=builder.compile(checkpointer=memory)

In [14]:
config1={'configurable':{'thread_id':"chat1"}}

res=graph.invoke({"messages":"what is ai?"},config=config1)

res

{'messages': [HumanMessage(content='what is ai?', additional_kwargs={}, response_metadata={}, id='ad2909d7-81e6-4b7e-bfbf-dfc4e1239d7e')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'model is about to answer users question', 'question': 'what is ai?', 'instructions': 'approve this question (y/n)?'}, id='435ccba51a69f46d6e71eeb07d20a1b5', response_schema=None)]}

In [16]:
final_res=graph.invoke(
    Command(
        resume={"approved":"y"}
    ),config=config1
)
final_res

{'messages': [HumanMessage(content='what is ai?', additional_kwargs={}, response_metadata={}, id='ad2909d7-81e6-4b7e-bfbf-dfc4e1239d7e'),
  AIMessage(content='**Artificial Intelligence (AI)** is a branch of computer science that aims to create machines (software, hardware, or both) capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and even creativity.\n\n---\n\n## Core Components\n\n| Component | What it does | Typical Techniques |\n|-----------|--------------|--------------------|\n| **Machine Learning (ML)** | Learns patterns from data without explicit programming. | Supervised/unsupervised learning, reinforcement learning, deep learning. |\n| **Knowledge Representation & Reasoning** | Stores facts and rules, then draws logical conclusions. | Ontologies, rule‑based systems, symbolic logic. |\n| **Natural Language Processing (NLP)** | Understands and generates human langua

In [17]:
config1={'configurable':{'thread_id':"chat1"}}

res=graph.invoke({"messages":"what is ai?"},config=config1)

res

{'messages': [HumanMessage(content='what is ai?', additional_kwargs={}, response_metadata={}, id='ad2909d7-81e6-4b7e-bfbf-dfc4e1239d7e'),
  AIMessage(content='**Artificial Intelligence (AI)** is a branch of computer science that aims to create machines (software, hardware, or both) capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and even creativity.\n\n---\n\n## Core Components\n\n| Component | What it does | Typical Techniques |\n|-----------|--------------|--------------------|\n| **Machine Learning (ML)** | Learns patterns from data without explicit programming. | Supervised/unsupervised learning, reinforcement learning, deep learning. |\n| **Knowledge Representation & Reasoning** | Stores facts and rules, then draws logical conclusions. | Ontologies, rule‑based systems, symbolic logic. |\n| **Natural Language Processing (NLP)** | Understands and generates human langua

In [18]:
final_res=graph.invoke(
    Command(
        resume={"approved":"n"}
    ),config=config1
)
final_res

{'messages': [HumanMessage(content='what is ai?', additional_kwargs={}, response_metadata={}, id='ad2909d7-81e6-4b7e-bfbf-dfc4e1239d7e'),
  AIMessage(content='**Artificial Intelligence (AI)** is a branch of computer science that aims to create machines (software, hardware, or both) capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and even creativity.\n\n---\n\n## Core Components\n\n| Component | What it does | Typical Techniques |\n|-----------|--------------|--------------------|\n| **Machine Learning (ML)** | Learns patterns from data without explicit programming. | Supervised/unsupervised learning, reinforcement learning, deep learning. |\n| **Knowledge Representation & Reasoning** | Stores facts and rules, then draws logical conclusions. | Ontologies, rule‑based systems, symbolic logic. |\n| **Natural Language Processing (NLP)** | Understands and generates human langua